# Local Spatial Autocorrelation

This notebook  focuses on local indicators of spatial association (LISA).

## Learning goals

By the end of this notebook, you will be able to:

- compute local Moran statistics for each areal unit
- interpret the Moran scatterplot quadrants
- identify statistically significant hot spots, cold spots, and spatial outliers
- distinguish global evidence of autocorrelation from local patterns

Local indicators of spatial association (LISA) answer the question that global Moran's $I$ cannot: **where** is the clustering occurring?


In [ ]:
import geopandas as gpd
import libpysal as lps
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import esda

Our data set comes from the Berlin airbnb scrape taken in April 2018. This dataframe was constructed as part of the [GeoPython 2018 workshop](https://github.com/ljwolf/geopython) by [Levi Wolf](https://ljwolf.org) and [Serge Rey](https://sergerey.org). As part of the workshop a geopandas data frame was constructed with one of the columns reporting the median listing price of units in each neighborhood in Berlin:

## Data preparation

We use the same Berlin neighborhood data and median Airbnb price variable as in the global analysis. Keeping the data and weights constant makes it easier to connect the local results back to the global Moran's $I$ statistic.


In [ ]:
gdf = gpd.read_file("data/berlin-neighbourhoods.geojson")

In [ ]:
bl_df = pd.read_csv("data/berlin-listings.csv")
geometry = gpd.points_from_xy(x=bl_df.longitude, y=bl_df.latitude, crs="epsg:4326")
bl_gdf = gpd.GeoDataFrame(bl_df, geometry=geometry)

In [ ]:
bl_gdf["price"] = bl_gdf["price"].astype("float32")
sj_gdf = gpd.sjoin(
    gdf, bl_gdf, how="inner", predicate="intersects", lsuffix="left", rsuffix="right"
)
median_price_gb = sj_gdf["price"].groupby([sj_gdf["neighbourhood_group"]]).mean()
median_price_gb

In [ ]:
gdf = gdf.join(median_price_gb, on="neighbourhood_group")
gdf.rename(columns={"price": "median_pri"}, inplace=True)
gdf.head(15)

We have an `nan` to first deal with:

In [ ]:
pd.isnull(gdf["median_pri"]).sum()

In [ ]:
gdf["median_pri"] = gdf["median_pri"].fillna(gdf["median_pri"].mean())

In [ ]:
gdf.plot(column="median_pri")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"aspect": "equal"})
gdf.plot(column="median_pri", scheme="Quantiles", k=5, cmap="GnBu", legend=True, ax=ax)
# ax.set_xlim(150000, 160000)
# ax.set_ylim(208000, 215000)

## From global to local autocorrelation

A significant global statistic tells us that the map is spatially structured overall, but it does not reveal which neighborhoods are driving that result. Local Moran statistics decompose that global pattern into unit-level contributions.


## Local Spatial Autocorrelation

In [ ]:
df = gdf
wq = lps.weights.Queen.from_dataframe(df, use_index=False, silence_warnings=True)
wq.transform = "r"

In [ ]:
y = df["median_pri"]
ylag = lps.weights.lag_spatial(wq, y)

The Moran scatterplot is a useful bridge between visualization and inference. It relates each neighborhood's value to the average value of its neighbors and organizes the observations into the familiar four quadrants of high-high, low-low, high-low, and low-high association.


## Local Autocorrelation: Hot Spots, Cold Spots, and Spatial Outliers
In addition to the Global autocorrelation statistics, PySAL has many local
autocorrelation statistics. Let's compute a local Moran statistic for the same
d

In [ ]:
np.random.seed(12345)

In [ ]:
wq.transform = "r"
lag_price = lps.weights.lag_spatial(wq, df["median_pri"])

In [ ]:
price = df["median_pri"]
b, a = np.polyfit(price, lag_price, 1)
f, ax = plt.subplots(1, figsize=(9, 9))

plt.plot(price, lag_price, ".", color="firebrick")

# dashed vert at mean of the price
plt.vlines(price.mean(), lag_price.min(), lag_price.max(), linestyle="--")
# dashed horizontal at mean of lagged price
plt.hlines(lag_price.mean(), price.min(), price.max(), linestyle="--")

# red line of best fit using global I as slope
plt.plot(price, a + b * price, "r")
plt.title("Moran Scatterplot")
plt.ylabel("Spatial Lag of Price")
plt.xlabel("Price")
plt.show()

Now, instead of a single $I$ statistic, we have an *array* of local $I_i$
statistics, stored in the `.Is` attribute, and p-values from the simulation are
in `p_sim`.

## Computing local Moran statistics

Each observation receives its own local statistic and permutation-based p-value. This allows us to distinguish places that are merely high or low in value from places that are embedded in statistically significant local structure.


In [ ]:
li = esda.moran.Moran_Local(y, wq)

We can again test for local clustering using permutations, but here we use
conditional random permutations (different distributions for each focal location)

In [ ]:
(li.p_sim < 0.05).sum()

## Interpreting cluster types

The quadrant labels correspond to substantively different local relationships:

- **High-High**: a hot spot
- **Low-Low**: a cold spot
- **High-Low**: a high-value spatial outlier
- **Low-High**: a low-value spatial outlier

In practice, we usually map only those observations that are statistically significant under the permutation test.


We can distinguish the specific type of local spatial association reflected in
the four quadrants of the Moran Scatterplot above:

In [ ]:
sig = li.p_sim < 0.05
hotspot = sig * li.q == 1
coldspot = sig * li.q == 3
doughnut = sig * li.q == 2
diamond = sig * li.q == 4

In [ ]:
spots = ["n.sig.", "hot spot"]
labels = [spots[i] for i in hotspot * 1]

In [ ]:
df = df
from matplotlib import colors

hmap = colors.ListedColormap(["red", "lightgrey"])
f, ax = plt.subplots(1, figsize=(9, 9))
df.assign(cl=labels).plot(
    column="cl",
    categorical=True,
    k=2,
    cmap=hmap,
    linewidth=0.1,
    ax=ax,
    edgecolor="white",
    legend=True,
)
ax.set_axis_off()
plt.show()

In [ ]:
spots = ["n.sig.", "cold spot"]
labels = [spots[i] for i in coldspot * 1]

In [ ]:
df = df
from matplotlib import colors

hmap = colors.ListedColormap(["blue", "lightgrey"])
f, ax = plt.subplots(1, figsize=(9, 9))
df.assign(cl=labels).plot(
    column="cl",
    categorical=True,
    k=2,
    cmap=hmap,
    linewidth=0.1,
    ax=ax,
    edgecolor="white",
    legend=True,
)
ax.set_axis_off()
plt.show()

In [ ]:
spots = ["n.sig.", "doughnut"]
labels = [spots[i] for i in doughnut * 1]

In [ ]:
df = df
from matplotlib import colors

hmap = colors.ListedColormap(["lightblue", "lightgrey"])
f, ax = plt.subplots(1, figsize=(9, 9))
df.assign(cl=labels).plot(
    column="cl",
    categorical=True,
    k=2,
    cmap=hmap,
    linewidth=0.1,
    ax=ax,
    edgecolor="white",
    legend=True,
)
ax.set_axis_off()
plt.show()

In [ ]:
spots = ["n.sig.", "diamond"]
labels = [spots[i] for i in diamond * 1]

In [ ]:
df = df
from matplotlib import colors

hmap = colors.ListedColormap(["pink", "lightgrey"])
f, ax = plt.subplots(1, figsize=(9, 9))
df.assign(cl=labels).plot(
    column="cl",
    categorical=True,
    k=2,
    cmap=hmap,
    linewidth=0.1,
    ax=ax,
    edgecolor="white",
    legend=True,
)
ax.set_axis_off()
plt.show()

In [ ]:
sig = 1 * (li.p_sim < 0.05)
hotspot = 1 * (sig * li.q == 1)
coldspot = 3 * (sig * li.q == 3)
doughnut = 2 * (sig * li.q == 2)
diamond = 4 * (sig * li.q == 4)
spots = hotspot + coldspot + doughnut + diamond
spots

In [ ]:
spot_labels = ["0 ns", "1 hot spot", "2 doughnut", "3 cold spot", "4 diamond"]
labels = [spot_labels[i] for i in spots]

In [ ]:
from matplotlib import colors

hmap = colors.ListedColormap(["lightgrey", "red", "lightblue", "blue", "pink"])
f, ax = plt.subplots(1, figsize=(9, 9))
df.assign(cl=labels).plot(
    column="cl",
    categorical=True,
    k=2,
    cmap=hmap,
    linewidth=0.1,
    ax=ax,
    edgecolor="white",
    legend=True,
)
ax.set_axis_off()
plt.show()

## Takeaways

Local Moran analysis moves from a single global summary to place-specific diagnostics. This is often the most interpretable stage of ESDA because it reveals where clustering and outliers are located on the map. At the same time, these local results should always be interpreted in the context of the global pattern, the choice of spatial weights, and the multiple-testing issues that arise when many local statistics are examined at once.
